# Soft targets and the counterfactual that reaches them

Pick a disease; get one figure with
1. **the targets**: where the low / mid / high soft targets sit in the C0 probability distribution of the class the counterfactual travels to (rotated, so its vertical axis is the probability),
2. **the run**: the guided diffusion trajectories converging on those same target lines, one per level,
3. **the result**: the original and the three counterfactuals.

Needs a GPU. Settings are the thesis' final diffusion settings (`tau = 150`, `scale = 150`, `FastDiME-woM`), the defaults of `code/cf_generate_levels.py`; the cell below checks that they still match.

In [ ]:
import os, sys, io, re, inspect
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

FE = "/zhome/d0/a/221493/thesis/final_experiment"     # absolute: the kernel's cwd is not this folder
sys.path.insert(0, f"{FE}/code")
from thesis_style import *
import cf_generation_final as cf
import cf_generate_levels as cgl
apply()
assert torch.cuda.is_available(), "no GPU visible to this kernel: start Jupyter on the GPU node"
assert cf.DEVICE == "cuda", f"cf.DEVICE is {cf.DEVICE}"
print("device", cf.DEVICE, torch.cuda.get_device_name(0))

# ── choose here ──────────────────────────────────────────────────────────────
DISEASE = "effusion"     # atelectasis | cardiomegaly | consolidation | edema | effusion
CELL    = "TN"           # confusion cell of the example image: TP | TN | FP | FN
PICK    = 3              # which image of that cell (any integer)
SEED    = 0              # diffusion noise draw

# ── final diffusion settings ─────────────────────────────────────────────────
TAU, SCALE, MODE = 150, 150, "FastDiME-woM"
_d = dict(re.findall(r'--(tau|scale)"[^)]*?default=(\d+)', inspect.getsource(cgl.main)))
assert (float(_d["tau"]), float(_d["scale"])) == (TAU, SCALE), f"script defaults changed: {_d}"

In [ ]:
unet, classifier, seg_model, cfg = cf.load_models(DISEASE)
df = pd.read_csv(cfg.c2_csv)
LEVELS = cgl.compute_levels(df, cfg.threshold, disease=DISEASE)

masks = {"TP": (df.pred == 1) & (df.true == 1), "TN": (df.pred == 0) & (df.true == 0),
         "FP": (df.pred == 1) & (df.true == 0), "FN": (df.pred == 0) & (df.true == 1)}
row = df[masks[CELL]].sample(1, random_state=PICK).iloc[0]
x0 = cf.load_image(row["path"])
p_orig = cf.get_c0_prob(x0, classifier)
direction = "to_positive" if cf.predict(p_orig, cfg) == 0 else "to_negative"
targets = LEVELS[direction]
print(f"{DISEASE}  {CELL}  p={p_orig:.3f}  threshold={cfg.threshold:.3f}  {direction}")
print("targets:", "  ".join(f"{n}={v:.3f}" for n, v in zip(cgl.LEVEL_NAMES, targets)))

In [ ]:
gradcam = cgl.GradCAM(classifier, cfg.target_idx, cfg.threshold)
sampler = cf.make_sampler(MODE, device=cf.DEVICE)
res = cgl.run_one(x0, direction, targets, unet, classifier, sampler, gradcam, cfg,
                  tau=TAU, scale=SCALE, seed=SEED, track_every=1)
for n, t, p in zip(cgl.LEVEL_NAMES, targets, res["level_probs"]):
    print(f"{n:5s} target {t:.3f}  reached {p:.3f}")

In [ ]:
C_NEG, C_POS = FAMILY["traj"], FAMILY["cnn"]          # green = negative, red = positive
LV_COL = RAMP["cnn"] if direction == "to_positive" else RAMP["traj"]   # light -> dark = low -> high

def plot_targets(res, targets, df, cfg, p_orig, direction, disease, tau, scale):
    fig = plt.figure(figsize=(FULL, 105 * MM))
    gs = GridSpec(2, 4, figure=fig, height_ratios=[1.2, 1], wspace=0.08, hspace=0.45)
    ax_t = fig.add_subplot(gs[0, 0:3])                     # trajectories
    ax_d = fig.add_subplot(gs[0, 3], sharey=ax_t)          # distribution, rotated
    thr = cfg.threshold

    # ── the distribution of the class the counterfactual travels to ──────────
    side, col = (1, C_POS) if direction == "to_positive" else (0, C_NEG)
    p = df.loc[(df["true"] == side) & ((df["prob"] > thr) if side else (df["prob"] < thr)), "prob"]
    dens, edges = np.histogram(p, bins=60, range=(0, 1), density=True)
    mid = (edges[:-1] + edges[1:]) / 2
    ax_d.fill_betweenx(mid, 0, dens, color=col, alpha=0.18, lw=0)
    ax_d.plot(dens, mid, color=col, lw=LW["ref"])
    ax_d.set_xticks([]); ax_d.spines["bottom"].set_visible(False)
    ax_d.tick_params(labelleft=False)
    ax_d.set_title("correctly classified\n" + ("positives" if side else "negatives"),
                   fontsize=FS["small"], loc="left")

    # ── targets: one line across both panels, in the level colour ────────────
    for ax in (ax_t, ax_d):
        ax.axhline(thr, color=INK, lw=LW["ref"], ls=(0, (3, 2)))
    for name, v, c in zip(cgl.LEVEL_NAMES, targets, LV_COL):
        ax_t.axhline(v, color=c, lw=LW["ref"], alpha=0.9)
        ax_d.axhline(v, color=c, lw=LW["ref"] * 1.6)
        ax_t.text(tau - 2, v, name, color=c, fontsize=FS["small"], va="bottom", ha="left")
    ax_t.text(tau - 2, thr, "threshold", color=INK, fontsize=FS["tiny"], va="bottom", ha="left")

    # ── the run: one guided trajectory per level ─────────────────────────────
    ax_t.axhline(p_orig, color=MUTED, lw=LW["hair"] * 2, ls=":")
    for i, c in enumerate(LV_COL):
        tj = res["trajs"][i]
        ax_t.plot(tj[:, 0], tj[:, 1], color=c, lw=LW["line"])
        ax_t.plot(1, res["level_probs"][i], "o", ms=4, color=c, mec=INK, mew=LW["hair"], zorder=5)
    ax_t.invert_xaxis()                                    # reverse process runs tau -> 1
    ax_t.set_xlim(tau, 0)
    ax_t.set_xlabel("diffusion timestep $t$  (denoising runs right to left)")
    ax_t.set_ylabel(f"$C_0$ probability of {disease}")
    ax_t.set_ylim(0, 1)
    ax_t.set_title(f"original $p={p_orig:.2f}$, {direction.replace('_', ' ')}", fontsize=FS["small"])

    # ── the result: original + three counterfactuals ─────────────────────────
    imgs = [res["x0"]] + res["levels"]
    titles = [f"original\n$p={p_orig:.2f}$"] + [
        f"{n}\ntarget {t:.2f}, got {pr:.2f}"
        for n, t, pr in zip(cgl.LEVEL_NAMES, targets, res["level_probs"])]
    for j, (img, t) in enumerate(zip(imgs, titles)):
        a = fig.add_subplot(gs[1, j])
        a.imshow(img, cmap="gray", vmin=0, vmax=1)
        a.set_xticks([]); a.set_yticks([])
        for s in a.spines.values():
            s.set_visible(True); s.set_linewidth(1.2 if j else 0.5)
            s.set_edgecolor(LV_COL[j - 1] if j else HAIR)
        a.set_title(t, fontsize=FS["small"])
    return fig

fig = plot_targets(res, targets, df, cfg, p_orig, direction, DISEASE, TAU, SCALE)
OUT = f"{FE}/viz/soft_targets_{DISEASE}"
fig.savefig(OUT + ".pdf"); fig.savefig(OUT + ".png", dpi=300)
plt.show()